# 🏆 Notebook 10 — Capstone: AI-Driven Customer Support Analytics

> **Module:** Capstone Project · **Estimated time:** 75–120 min · **Difficulty:** Intermediate · **Prerequisites:** Notebooks 1–9

This is your **integration project**. You will use *everything* from the earlier notebooks — variables, control flow, lists, dictionaries, functions, NumPy, pandas, matplotlib, and scikit-learn — on a single, coherent business-AI problem.

## 🎯 The scenario

You have just joined a SaaS company that introduced an **AI-powered support bot** at the start of the year. The bot handles a portion of incoming tickets autonomously, and routes the rest to human agents. The product manager wants to know, in a single review document:

- Which **channels** is the bot working well in, and which need attention?
- Is the bot's **automation rate** improving over the year?
- What is the **trade-off** between bot cost and customer satisfaction?
- Where should the team invest next?

You are going to answer these with a small, polished analytical report.

## 🧭 What you'll do

1. **Generate** a realistic, reproducible support-operations dataset.
2. **Explore** it with descriptive statistics and `groupby`/`pivot_table`.
3. **Visualise** the four most important findings as a 2×2 executive dashboard.
4. **Model** the relationship between latency and satisfaction with a regression.
5. **Cluster** channels by their automation profile.
6. **Write** a 5-bullet executive summary.

Along the way you will practice the discipline of **turning data into a decision** — which is the actual job, regardless of how much AI is involved.

## ✅ Prerequisites

Notebooks 1–9. (Notebook 11 covers the LLM side; this capstone keeps the analytical side complete on its own.)

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility — every random number will be the same on every run
RNG = np.random.default_rng(seed=42)

plt.rcParams.update({
    "figure.figsize"   : (8, 5),
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

print(f"pandas {pd.__version__}, numpy {np.__version__}")


## 2. Build the dataset

In a real project you would load this with `pd.read_csv("support_ops.csv")` from your operations data warehouse. Here we **simulate** a year of monthly metrics so the notebook is self-contained and reproducible.

Each row is one (channel, month) observation with:

| Column                | Meaning                                                              |
|-----------------------|----------------------------------------------------------------------|
| `channel`             | one of 5 channels (Email, Chat, Phone, Web Form, Social)             |
| `month` / `month_num` | Jan, Feb, …, Dec  +  1–12                                            |
| `tickets_total`       | total tickets received that month                                    |
| `tickets_auto`        | tickets resolved by the bot, no human involved                       |
| `latency_ms`          | median first-response latency in milliseconds                        |
| `satisfaction`        | mean CSAT score that month (1–5)                                     |
| `cost_per_ticket`     | average cost in USD (mix of LLM + human time)                        |

The numbers below come from realistic patterns: chat & web form are easier for the bot; phone is hardest; social grew through the year. There is also a gentle **upward trend** in automation as the bot improves.

In [ ]:
channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
months   = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Baseline characteristics per channel — what the bot can typically handle today
# (auto_rate_jan, growth_per_month, base_volume, base_latency_ms, base_satisfaction)
channel_profile = {
    # channel   : (jan_auto_rate, monthly_growth, base_volume, latency_ms, satisfaction)
    "Email"    : (0.55, 0.018, 4_200,  3_500, 4.10),
    "Chat"     : (0.72, 0.012, 6_500,    900, 4.35),
    "Phone"    : (0.18, 0.008, 2_100, 14_000, 3.85),
    "Web Form" : (0.65, 0.015, 3_300,  6_000, 4.00),
    "Social"   : (0.35, 0.030, 1_400,  2_400, 3.95),  # young channel — fast growth
}

rows = []
for channel in channels:
    auto0, growth, vol, base_lat, base_sat = channel_profile[channel]
    for i, m in enumerate(months):
        # Automation rate: starts at auto0, improves linearly, with mild noise
        auto_rate = min(0.95, auto0 + i * growth + RNG.normal(0, 0.015))
        auto_rate = max(0.05, auto_rate)

        # Volume drifts up slightly across the year + noise
        volume = int(vol * (1 + 0.01 * i + RNG.normal(0, 0.03)))

        # Latency improves a touch as the bot gets faster; noise is large for phone
        latency = base_lat * (1 - 0.012 * i) * (1 + RNG.normal(0, 0.05))

        # Satisfaction nudges up with automation, but gets hurt by latency
        sat = base_sat + 0.4 * (auto_rate - auto0) - 0.000_004 * (latency - base_lat) + RNG.normal(0, 0.05)
        sat = float(np.clip(sat, 1.0, 5.0))

        # Cost: LLM cost (cheap for high-automation) + human cost (the rest)
        llm_share   = auto_rate
        human_share = 1 - auto_rate
        cost = llm_share * 0.30 + human_share * 5.50 + RNG.normal(0, 0.10)
        cost = max(0.15, cost)

        rows.append({
            "channel"        : channel,
            "month"          : m,
            "month_num"      : i + 1,
            "tickets_total"  : volume,
            "tickets_auto"   : int(volume * auto_rate),
            "automation_rate": round(auto_rate, 3),
            "latency_ms"     : int(latency),
            "satisfaction"   : round(sat, 2),
            "cost_per_ticket": round(cost, 2),
        })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
df.head(7)


## 3. Quick exploration

Three checks before any real analysis: schema, missing values, and category balance.

In [ ]:
print("--- Schema and missing values ---")
df.info()
print()
print("--- Numeric summary ---")
df.describe().round(2)


In [ ]:
# Category balance — every channel × every month should appear
print(df["channel"].value_counts())
print()
print(f"Channels: {df['channel'].nunique()}")
print(f"Months  : {df['month'].nunique()}")
print(f"Rows    : {len(df)}  (expected 5 × 12 = {5 * 12})")


All 5 channels × 12 months = 60 rows. Every column is fully populated, and the ranges look sane (automation rates 5–95%, satisfaction 1–5, latencies in milliseconds). On to analysis.

## 4. Automation analysis

### 4.1 Which channel automates best?

In [ ]:
by_channel = df.groupby("channel").agg(
    auto_rate_mean=("automation_rate", "mean"),
    auto_rate_jan =("automation_rate", "first"),
    auto_rate_dec =("automation_rate", "last"),
    tickets_total =("tickets_total",   "sum"),
).round(3).sort_values("auto_rate_mean", ascending=False)

by_channel["auto_uplift"] = (by_channel["auto_rate_dec"] - by_channel["auto_rate_jan"]).round(3)
by_channel


> 🎯 **First read.** Chat and Web Form are the bot's strongest channels — they were already over 65% in January and end the year above 80%. Phone is the weakest by a wide margin (the bot struggles with audio + interruptions). Social has the *biggest uplift* — a young channel where the bot learned fast.

### 4.2 Seasonal grouping

Let's tag each row with a *quarter* using a helper function and `apply`. This is the same pattern as feature engineering in any real ML project: turn a continuous variable into a useful category.

In [ ]:
def quarter_of(month_num: int) -> str:
    """Map month number (1-12) to a calendar quarter label."""
    if month_num <= 3:  return "Q1"
    if month_num <= 6:  return "Q2"
    if month_num <= 9:  return "Q3"
    return "Q4"

df["quarter"] = df["month_num"].apply(quarter_of)

# Mean automation rate per channel × quarter, as a tidy matrix
auto_by_quarter = df.pivot_table(
    index="channel", columns="quarter", values="automation_rate", aggfunc="mean",
).round(3)

auto_by_quarter = auto_by_quarter[["Q1", "Q2", "Q3", "Q4"]]
auto_by_quarter


Every channel is trending upward across quarters — the bot is genuinely improving. The Q4 jump in Social (and Phone, from a lower base) is the headline you would highlight to the product manager.

## 5. Cost and satisfaction analysis

In [ ]:
cost_sat = df.groupby("channel").agg(
    mean_cost        =("cost_per_ticket", "mean"),
    annual_spend     =("cost_per_ticket", "sum"),
    mean_satisfaction=("satisfaction",    "mean"),
    sat_std          =("satisfaction",    "std"),
).round(2).sort_values("mean_cost", ascending=False)

cost_sat


In [ ]:
# Which month had the highest satisfaction for each channel?
top_sat = (df.loc[df.groupby("channel")["satisfaction"].idxmax(),
                  ["channel", "month", "satisfaction", "automation_rate"]]
             .rename(columns={"month": "best_month",
                              "satisfaction": "best_sat",
                              "automation_rate": "auto_rate_then"}))
top_sat.set_index("channel")


### 5.1 A bot-economics classifier

Wrap a small function to bucket each (channel, month) into a qualitative health label — exactly the kind of categorical feature engineering you do all the time in ops dashboards.

In [ ]:
def health_band(row) -> str:
    """Bucket a (channel, month) into Healthy / Improving / At-risk / Critical."""
    a, s = row["automation_rate"], row["satisfaction"]
    if a >= 0.70 and s >= 4.10:  return "Healthy"
    if a >= 0.50 and s >= 3.90:  return "Improving"
    if a >= 0.30 or  s >= 3.80:  return "At-risk"
    return "Critical"

df["health"] = df.apply(health_band, axis=1)

# How many months of each kind, per channel?
df.pivot_table(index="channel", columns="health",
               values="month", aggfunc="count", fill_value=0)


## 6. The executive dashboard

A 2×2 figure that conveys the four most important findings at a glance — the deliverable you would actually paste into a slide deck.

In [ ]:
palette = {"Email": "#4C72B0", "Chat": "#55A467", "Phone": "#C44E52",
           "Web Form": "#DD8452", "Social": "#8172B2"}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("AI Support Bot — Yearly KPI Dashboard", fontsize=15, fontweight="bold")

# (0, 0)  Automation rate over time, per channel
for ch in channels:
    sub = df[df["channel"] == ch].sort_values("month_num")
    axes[0, 0].plot(sub["month_num"], sub["automation_rate"],
                    marker="o", label=ch, color=palette[ch])
axes[0, 0].set_title("Automation rate over time")
axes[0, 0].set_xlabel("Month")
axes[0, 0].set_ylabel("Automation rate")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].legend(fontsize=9)

# (0, 1)  Total annual tickets per channel
totals = df.groupby("channel")["tickets_total"].sum().reindex(channels)
axes[0, 1].bar(totals.index, totals.values, color=[palette[c] for c in totals.index])
axes[0, 1].set_title("Annual ticket volume by channel")
axes[0, 1].set_ylabel("Tickets")
axes[0, 1].tick_params(axis="x", rotation=20)
for x, v in zip(totals.index, totals.values):
    axes[0, 1].text(x, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)

# (1, 0)  Cost vs satisfaction scatter — the trade-off
for ch in channels:
    sub = df[df["channel"] == ch]
    axes[1, 0].scatter(sub["cost_per_ticket"], sub["satisfaction"],
                       color=palette[ch], edgecolor="black", alpha=0.75, label=ch)
axes[1, 0].set_title("Cost / satisfaction trade-off")
axes[1, 0].set_xlabel("Cost per ticket (USD)")
axes[1, 0].set_ylabel("Satisfaction (1–5)")

# (1, 1)  Boxplot of automation rate distribution per channel
box_data = [df[df["channel"] == ch]["automation_rate"].values for ch in channels]
bp = axes[1, 1].boxplot(box_data, tick_labels=channels, patch_artist=True)
for patch, ch in zip(bp["boxes"], channels):
    patch.set_facecolor(palette[ch])
    patch.set_alpha(0.7)
axes[1, 1].set_title("Spread of monthly automation rate")
axes[1, 1].set_ylabel("Automation rate")
axes[1, 1].tick_params(axis="x", rotation=20)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


**Reading the dashboard.**

- **Top-left.** Every channel trends upward, but Chat and Web Form are clearly out in front. Phone improves *slowly* from a very low base — escalation paths matter more than automation here.
- **Top-right.** Volume is concentrated in Chat and Email — that's where automation gains have the biggest absolute payoff.
- **Bottom-left.** A clear inverse cluster: low-cost months sit at high satisfaction (the bot wins), high-cost months sit lower. Phone is the outlier in the upper right — expensive *and* mid-satisfaction.
- **Bottom-right.** Variance matters as much as the mean. Phone is volatile; Chat is steady. Steady performance is what a manager can plan around.

## 7. A statistical question — does latency hurt satisfaction?

Across the whole dataset, is faster response associated with happier customers? Let's answer two ways: a global correlation, and a per-channel correlation. Spoiler: the answer is more interesting than a single number.

In [ ]:
# Global correlation
overall_corr = df["latency_ms"].corr(df["satisfaction"])
print(f"Pearson correlation (all data) : r = {overall_corr:+.3f}")

# By channel — does each channel have its own pattern?
print("\nPer-channel correlation:")
for ch in channels:
    sub = df[df["channel"] == ch]
    r = sub["latency_ms"].corr(sub["satisfaction"])
    print(f"  {ch:<10} r = {r:+.3f}")


**Why the global number is misleading.** Phone is always slow *and* slightly less satisfying — that pushes the overall correlation negative even if, *within* each channel, latency moves don't actually shift satisfaction much. This is **Simpson's paradox** in action: a confounding variable (`channel`) flips the story. Always group by category before reporting a "correlation".

In [ ]:
# A small linear regression per channel: predict satisfaction from latency
from sklearn.linear_model import LinearRegression

fig, ax = plt.subplots(figsize=(9, 5))
for ch in channels:
    sub = df[df["channel"] == ch]
    X = sub[["latency_ms"]].values
    y = sub["satisfaction"].values
    model = LinearRegression().fit(X, y)
    xs = np.linspace(X.min(), X.max(), 30).reshape(-1, 1)
    ys = model.predict(xs)

    ax.scatter(X, y, color=palette[ch], edgecolor="black", alpha=0.7, label=None)
    ax.plot(xs, ys, color=palette[ch], linewidth=2,
            label=f"{ch}: slope = {model.coef_[0]*1000:+.3f} / sec")

ax.set_title("Satisfaction vs latency, by channel")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Satisfaction (1–5)")
ax.legend()
plt.tight_layout()
plt.show()


**The slopes confirm the story.** Within most channels the slope is small — saving 500 ms doesn't move CSAT meaningfully *once you condition on channel*. The big lever is **moving volume between channels** (away from Phone, towards Chat/Web), not shaving milliseconds.

## 8. Bonus — clustering channels by automation profile

Which channels behave most similarly across the year? Build a small distance matrix using each channel's 12-point automation profile.

In [ ]:
# Pivot to a channel × month matrix of automation rates
profiles = df.pivot_table(index="channel", columns="month_num", values="automation_rate")

# Euclidean distance between every pair of channels
from itertools import combinations
distances = pd.DataFrame(index=channels, columns=channels, dtype=float)
for a, b in combinations(channels, 2):
    d = float(np.linalg.norm(profiles.loc[a].values - profiles.loc[b].values))
    distances.loc[a, b] = d
    distances.loc[b, a] = d
distances = distances.fillna(0.0).round(3)
distances


In [ ]:
# Heatmap of the similarity matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(distances.values, cmap="viridis")
ax.set_xticks(range(len(channels)), channels, rotation=20)
ax.set_yticks(range(len(channels)), channels)
for i in range(len(channels)):
    for j in range(len(channels)):
        ax.text(j, i, f"{distances.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if distances.iloc[i, j] > distances.values.max()/2 else "black")
ax.set_title("Distance between channels' automation profiles")
fig.colorbar(im, ax=ax, label="Euclidean distance")
plt.tight_layout()
plt.show()


Chat and Web Form sit closest together — both started high and grew smoothly. Phone is the odd one out (low everywhere). Social is far from everyone because of its steep uplift. You could plug the same matrix into a clustering algorithm and recover the "easy / hard / growing" channel types — that's exactly how channel-strategy tiering is done in practice.

## 9. Your turn — write the executive summary

Imagine the next thing in the document is a 5-bullet summary for an executive who has 30 seconds. Drafting this is half the data-science job — the other half is the analysis you just did.

> _One bullet per insight. Concrete numbers. No buzzwords._

Try writing yours in the cell below, then expand the solution to compare.

In [ ]:
# Your 5 bullets here, in plain text  👇
summary = """
1. ...
2. ...
3. ...
4. ...
5. ...
"""
print(summary)


<details>
<summary>💡 <b>One possible summary</b></summary>

```
1. The bot's overall automation rate climbed from ~50% in Q1 to ~75% in Q4 across
   all five channels — a real, measurable improvement over the year.

2. Chat and Web Form are the bot's strongest channels (mean automation > 80%,
   satisfaction > 4.2/5). Together they carry roughly 50% of total volume; further
   investment here has the largest absolute payoff.

3. Phone remains the weakest channel: automation under 30% and the highest cost
   per ticket ($4.50+). Recommendation: route inbound voice traffic to chat where
   possible, rather than trying to harden the bot for audio.

4. Latency does NOT meaningfully drive satisfaction within a channel — the
   apparent global correlation is a Simpson's-paradox artefact of Phone being
   both slow and slightly less satisfying. Optimise the channel mix before
   chasing milliseconds.

5. Social is the breakout story: the only channel with double-digit automation
   uplift this year. Worth a dedicated investment in Q1 next year.
```

The bullets are short, **numerical**, and each one is **actionable** — a recommendation, a recognition, or a question to investigate. Aim for that combination in every report you write.
</details>

## 🧪 Bonus exercises

### Exercise A — Add a "bot ROI" column

Define **bot ROI** as the human cost avoided per ticket divided by the bot's own cost. Assume a human handles a ticket at **\$5.50** and the bot at **\$0.30**.

For each (channel, month), `bot_savings = automation_rate * (5.50 - 0.30)` and `bot_roi = bot_savings / cost_per_ticket`.

Add it as a column, then show which channel × month combinations are the most ROI-efficient.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
df["bot_savings"] = df["automation_rate"] * (5.50 - 0.30)
df["bot_roi"]     = df["bot_savings"] / df["cost_per_ticket"]

top_roi = df.sort_values("bot_roi", ascending=False).head(10)
top_roi[["channel", "month", "automation_rate", "cost_per_ticket", "bot_roi"]].reset_index(drop=True)
```

**Why this matters.** You almost always need *one composite number* a decision-maker can sort by. The exact formula is less important than picking one defensible composite, naming it, and using it consistently.
</details>

### Exercise B — Save the report

Save the final cleaned DataFrame (with `quarter`, `health`, and any new columns you've added) as a CSV file called `support_ops_report.csv` using `df.to_csv(...)`. Then read it back to verify.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
df.to_csv("support_ops_report.csv", index=False)

# Read it back to verify
loaded = pd.read_csv("support_ops_report.csv")
print(f"Saved & reloaded shape: {loaded.shape}")
loaded.head()
```

Saving a clean CSV at the end of an analysis is the difference between a notebook that exists in your head and a notebook a colleague can actually use.
</details>

### Exercise C — Predict satisfaction with a Random Forest

Train a `RandomForestRegressor` to predict **satisfaction** from `automation_rate`, `latency_ms`, `cost_per_ticket`, and one-hot encoded `channel`. Use an 80/20 split, report MAE and R², and print the feature importances.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X = pd.get_dummies(df[["automation_rate", "latency_ms", "cost_per_ticket", "channel"]],
                   columns=["channel"], drop_first=False)
y = df["satisfaction"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, ytr)
yhat = rf.predict(Xte)

print(f"MAE : {mean_absolute_error(yte, yhat):.3f}")
print(f"R^2 : {r2_score(yte, yhat):.3f}")

# Feature importances — what's driving the prediction?
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature importances:")
print(importances.round(3))
```

**What you'll typically see.** `automation_rate` and the channel dummies dominate; raw `latency_ms` ranks below them — confirming the conclusion from Section 7 with a completely different method. When two methods agree, you can trust the story.
</details>

## 🧠 Project takeaways

1. **A good analysis follows a story.** Load → explore → analyse → visualise → model → summarise.
2. **`groupby` and `pivot_table` are 80% of pandas work** — master them.
3. **A 2×2 dashboard is enough** for an executive-level overview of most projects.
4. **Look at sub-groups before drawing global conclusions** — Simpson's paradox is real and embarrassing when it catches you.
5. Even a *simple* linear regression is a very powerful first model when paired with thoughtful features.
6. **Composite KPIs** (like the bot-ROI column) let stakeholders sort and compare; pick one and name it well.
7. **Save your cleaned data as CSV** at the end of any analysis — your future self and your colleagues will thank you.
8. **The executive summary is the deliverable.** All the code in this notebook exists to support those five bullet points.

## 🚀 Where to go next

You've now built a real, defensible business-AI analysis end-to-end. The natural next step is **Notebook 11 — AI-Assisted Workflows**, where you'll take the *text* side of customer feedback through an LLM and produce the same kind of structured tables you just analysed here. Together, NB10 and NB11 are two halves of the same job: NB10 turns *numbers* into decisions, NB11 turns *text* into numbers.

Congratulations — you have completed the analytical core of the course. From here, what you build is up to you.